# HomeValue.AI - Classification de l'état d'un bien (`condition`)

**Objectif.** Prédire `condition` (1 à 5) à partir des caractéristiques d'un bien immobilier (King County).

**Démarche scientifique suivie dans ce notebook :**
1. Contrôle qualité et analyse de la **distribution des classes**.
2. Split **stratifié** train/test (le test reste un hold-out honnête, jamais touché avant l'évaluation finale),comme nous l'a appris M. Abdouraman😁.
3. Rééquilibrage : comparaison rigoureuse **par validation croisée** de 3 stratégies (rien / `class_weight` / **SMOTE**). Le sur-échantillonnage est fait *dans un pipeline*  uniquement sur les plis d'entraînement (pas de data leakage).
4. Ajustement des hyperparamètres par CV(Cross Validation...biensur).
5. **Calibration** des probabilités (l'application affiche une distribution de probabilités) + analyse du compromis calibration vs rappel des minorités.
6. Discussion des **seuils de décision** et de la **réduction de dimension**. Normalement, ca se fait avec le metier, mais bon, on fera sans.
7. Évaluation finale  et export du bundle `.pkl` pour notre API.

## 1. Importation et contrôle qualité

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data_init = pd.read_csv('kc_house_data.csv')
data_init.head()

In [ ]:
# id inutile pour la prédiction, on vire ça très vite 
data = data_init.drop('id', axis=1)
data.dtypes

In [ ]:
# Qualité des données : pas de valeurs manquantes, on vérifie aussi les doublons
print('Valeurs manquantes :', data.isna().sum().sum())
print('Doublons          :', data.duplicated().sum())

## 2. Corrélations et colinéarité

On retire une variable fortement corrélée pour éviter la redondance.

In [ ]:
plt.figure(figsize=(12, 8))
num = data.select_dtypes(include=[np.number]).columns # On sélectionne les colonnes numériques pour la corrélation
sns.heatmap(data[num].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.show()

In [ ]:
# sqft_above ~ colinéaire avec sqft_living (corr très élevée) -> on la supprime
data = data.drop('sqft_above', axis=1)

## 3. Analyse de la variable cible `condition`

C'est le point le plus important du problème : la cible est **très déséquilibrée**.

In [ ]:
counts = data['condition'].value_counts().sort_index()
pct = (counts / len(data) * 100).round(2)
pd.DataFrame({'effectif': counts, 'part_%': pct})

In [ ]:
sns.barplot(x=counts.index, y=counts.values, color='lightblue')
plt.title('Distribution de condition'); plt.xlabel('condition'); plt.ylabel('effectif')
plt.show()

**Observation (déterminante pour toute la calibration dans la suite).**

| condition | effectif | part |
|---|---|---|
| 1 | 30 | 0.14 % |
| 2 | 172 | 0.80 % |
| 3 | 14 031 | 64.9 % |
| 4 | 5 679 | 26.3 % |
| 5 | 1 701 | 7.9 % |

- Les classes **1 et 2 sont quasi inapprenables** : 30 et 172 exemples au total. Après split, le test ne contient que qu'environ 6 et eniron 34 cas. **Aucune méthode ne « répare » 30 exemples** ; l'honnêteté impose de le dire plutôt que d'afficher un faux score après avoir forcé.
- Le déséquilibre impose : (a) une **métrique adaptée** (`f1_macro`, `balanced_accuracy`, qui pèsent chaque classe également), pas l'accuracy simple ; (b) une **stratégie de rééquilibrage** évaluée proprement ; (c) une **stratification** systématique des splits.




## 4. Feature engineering

On extrait l'année de vente pour calculer l'âge du bien, et on binarise la rénovation.

In [ ]:
data['date'] = pd.to_datetime(data['date'], format='%Y%m%dT%H%M%S')
data['yr_sold'] = data['date'].dt.year
data['house_age'] = data['yr_sold'] - data['yr_built'] # Annee de vente - annee de construction = age de la maison
data['was_renovated'] = (data['yr_renovated'] > 0).astype(int)
data = data.drop(['date', 'yr_built', 'yr_renovated'], axis=1)
data.head()

## 5. Split stratifié train / test

Le test est mis de côté comme **hold-out final** et n'intervient dans aucun choix de modèle.

In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop('condition', axis=1)
y = data['condition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

## 6. Rééquilibrage : comparaison rigoureuse des stratégies

On compare **par validation croisée stratifiée** (5 plis, sur le *train* seulement) trois stratégies :
- forêt **brute**,
- forêt avec **`class_weight='balanced'`** (ce qu'utilisait l'ancien modèle),
- **SMOTE + forêt**, où SMOTE est dans un `Pipeline` imblearn : il ne sur-échantillonne **que le pli d'entraînement** de chaque fold (aucune fuite vers le pli de validation).

Métriques : `f1_macro` et `balanced_accuracy` (adaptées au déséquilibre).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['f1_macro', 'balanced_accuracy']

strategies = {
    'RF brute':      RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    'RF class_weight': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1),
    'SMOTE + RF':    ImbPipeline([('smote', SMOTE(random_state=42)),
                                  ('rf', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))]),
}

rows = []
for name, model in strategies.items():
    r = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({'strategie': name,
                 'f1_macro': r['test_f1_macro'].mean(),
                 'balanced_acc': r['test_balanced_accuracy'].mean()})
pd.DataFrame(rows).set_index('strategie').round(3)

**Observation.** SMOTE domine nettement sur les deux métriques (`f1_macro` ≈ 0.37, `balanced_acc` ≈ 0.37) contre ≈ 0.31 pour les deux forêts non ré-échantillonnées.

Point notable : **`class_weight='balanced'` est la *pire* stratégie**. On retient donc **SMOTE + forêt**.

## 7. Ajustement des hyperparamètres (CV)

On affine la forêt à l'intérieur du pipeline SMOTE, toujours par CV `f1_macro`.

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe = ImbPipeline([('smote', SMOTE(random_state=42)),
                    ('rf', RandomForestClassifier(random_state=42, n_jobs=-1))])

param_grid = {
    'rf__n_estimators': [200, 300,400,500],
    'rf__max_depth': [20, 30,50,None],
    'rf__min_samples_leaf': [1, 2, 4], # Bon, on va garder une valeur basse pour ne pas trop lisser les classes rares (condition 1 et 2)
}
grid = GridSearchCV(pipe, param_grid, cv=3, scoring='f1_macro', n_jobs=-1)
grid.fit(X_train, y_train)
grid.best_params_

## 8. Calibration des probabilités

L'application affiche une **distribution de probabilités** comme si elle était fiable. Une forêt (surtout après SMOTE, qui déforme les proportions) n'est **pas calibrée** : ses probabilités ne reflètent pas les fréquences réelles. On calibre donc le modèle et on **mesure** l'effet — on ne l'applique pas à l'aveugle.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss, balanced_accuracy_score, f1_score

best = grid.best_estimator_
best.fit(X_train, y_train)

calibrated = CalibratedClassifierCV(best, method='sigmoid', cv=3)
calibrated.fit(X_train, y_train)

def scores(model, tag):
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)
    return {'modele': tag,
            'log_loss': log_loss(y_test, proba, labels=list(model.classes_)),
            'f1_macro': f1_score(y_test, pred, average='macro'),
            'balanced_acc': balanced_accuracy_score(y_test, pred)}

pd.DataFrame([scores(best, 'SMOTE+RF (non calibré)'),
              scores(calibrated, 'SMOTE+RF calibré')]).set_index('modele').round(3)

**Observation  un vrai compromis à documenter.**

- La calibration **améliore le `log_loss`** (probabilités plus honnêtes) : c'est ce qui compte pour un affichage de probabilités.
- Mais elle **ramène les décisions `argmax` vers la classe majoritaire**, donc `balanced_accuracy` et `f1_macro` baissent : en devenant honnêtes, les probabilités cessent de sur-parier sur les minorités.

**Décision assumée.** Comme le produit *montre* les probabilités à l'utilisateur, afficher des probabilités mal calibrées et sur-confiantes serait trompeur. On **déploie le modèle calibré** (meilleur `log_loss`) et on documente que, si l'objectif prioritaire était le rappel des minorités, on garderait la version non calibrée (meilleur `balanced_acc`).

In [ ]:
from sklearn.calibration import calibration_curve

# Fiabilité pour la classe majoritaire (3) : proba prédite vs fréquence observée
c = calibrated.classes_.tolist().index(3)
prob_uncal = best.predict_proba(X_test)[:, c]
prob_cal = calibrated.predict_proba(X_test)[:, c]
y3 = (y_test == 3).astype(int)

plt.figure(figsize=(6, 6))
for p, lab in [(prob_uncal, 'non calibré'), (prob_cal, 'calibré')]:
    frac, mean = calibration_curve(y3, p, n_bins=10)
    plt.plot(mean, frac, marker='o', label=lab)
plt.plot([0, 1], [0, 1], 'k--', label='idéal')
plt.xlabel('probabilité prédite'); plt.ylabel('fréquence observée')
plt.title('Courbe de fiabilité — classe 3'); plt.legend()
plt.show()

## 9. Seuils de décision

En multiclasses, la règle par défaut est l'`argmax` des probabilités. On pourrait remplacer cela par une règle **coût-sensible** (`argmax p_k / prior_k`) pour re-sensibiliser aux minorités, ou par des seuils par classe.

**Choix.** On garde l'`argmax` sur les probabilités **calibrées**, pour deux raisons : (a) toucher aux seuils reviendrait à annuler la calibration qu'on vient de justifier ; (b) sur les classes 1–2, aucun seuil ne compense l'absence de données. Ajuster un seuil ici donnerait l'illusion d'un gain sans réalité statistique. Le levier décisif reste **plus de données** sur les états rares, pas un réglage de seuil.

## 10. Réduction de dimension

**Décision : on n'applique pas de PCA.** C'est un choix raisonné, pas un oubli :
- 18 features seulement, toutes interprétables  pas de fléau de la dimension.
- Une forêt aléatoire est robuste aux variables peu informatives (elle les ignore par les splits) ; la PCA n'améliore pas les arbres et **détruit l'interprétabilité** (l'app explique un résultat métier).
- La colinéarité gênante a déjà été traitée (`sqft_above` retiré). 
En bref j'ai pensé a la reduction de dimension, mais j'ai conclu qu'ici c'est inutile. 

On préfère l'**importance des variables** (section suivante) comme outil de sélection interprétable.

## 11. Évaluation finale sur le hold-out

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

y_pred = calibrated.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))
print('balanced_accuracy :', round(balanced_accuracy_score(y_test, y_pred), 3))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues', colorbar=False, normalize='true')
plt.title('Matrice de confusion normalisée — condition'); plt.tight_layout(); plt.show()

**Lecture honnête.** Le modèle sépare correctement les classes 3/4/5 ; les classes 1 et 2 restent à ~0 de rappel — conforme au diagnostic initial (30 et 172 exemples). Ce n'est pas un défaut du modèle mais une **limite des données** : le résultat est présenté tel quel, sans le maquiller.

In [ ]:
importances = pd.Series(best.named_steps['rf'].feature_importances_, index=X_train.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=importances.values, y=importances.index)
plt.title('Importance des variables (forêt)'); plt.show()

## 12. Export du bundle `.pkl`

On ré-entraîne le modèle final retenu sur **tout** le jeu de données (train + test) pour la mise en production, et on sérialise le bundle consommé par le backend. `train.py` reproduit exactement ce pipeline en ligne de commande.

In [ ]:
import pickle
from pathlib import Path

FEATURE_ORDER = ['price','bedrooms','bathrooms','sqft_living','sqft_lot','floors',
                 'waterfront','view','grade','sqft_basement','zipcode','lat','long',
                 'sqft_living15','sqft_lot15','yr_sold','house_age','was_renovated']

final_pipe = ImbPipeline([('smote', SMOTE(random_state=42)),
                          ('rf', RandomForestClassifier(**{k.replace('rf__',''): v for k, v in grid.best_params_.items()},
                                                        random_state=42, n_jobs=-1))])
final_model = CalibratedClassifierCV(final_pipe, method='sigmoid', cv=3)
final_model.fit(X[FEATURE_ORDER], y)

bundle = {
    'model': final_model,
    'features': FEATURE_ORDER,
    'classes': [int(c) for c in final_model.classes_],
    'target': 'condition',
    'metrics': {'f1_macro': round(float(f1_score(y_test, y_pred, average='macro')), 4),
                'balanced_accuracy': round(float(balanced_accuracy_score(y_test, y_pred)), 4)},
}
out = Path('ml/artifacts/model.pkl')
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, 'wb') as f:
    pickle.dump(bundle, f)
print('Modele exporte ->', out)